In [42]:
from google.cloud import storage
import os
import torch

import sys
from pathlib import Path

ROOT = Path("./arcface_torch")
sys.path.insert(0, str(ROOT))

from backbones import get_model
from torch.distributed.algorithms.ddp_comm_hooks.default_hooks import fp16_compress_hook
from torch import distributed

from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
import pickle
from collections import Counter
from torch.utils.data import Dataset
import random
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
import torch
from PIL import Image
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
import numpy as np
np.long = np.int_
np.ulong = np.uint

if not hasattr(np, "bool"):
    np.bool = bool
    
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.model_selection import KFold

### Download Last Checkpoint

In [2]:
# LAST_CHECKPOINT = "gs://rf-pytorch-bucket/artifacts/20261020T125336278692Z/checkpoint_gpu_0.pt"
# bucket_name, blob_name = LAST_CHECKPOINT[5:].split("/", 1)
# client = storage.Client()
# bucket = client.bucket(bucket_name)
# blob = bucket.blob(blob_name)
# checkpoint_path = os.path.join(
#     "work_dirs",
#     f"checkpoint_gpu_0.pt",
# )
# blob.download_to_filename(checkpoint_path)
# checkpoint = torch.load(
#     checkpoint_path,
#     weights_only=False,
# )

In [3]:
checkpoint_path = os.path.join(
    "work_dirs",
    f"checkpoint_gpu_0.pt",
)

In [4]:
checkpoint = torch.load(
    checkpoint_path,
    weights_only=False,
)

### Load Model

In [1]:
# rank = 0
# local_rank = 0
# world_size = 1

# distributed.init_process_group(
#     backend="nccl",
#     init_method="tcp://127.0.0.1:12583",
#     rank=rank,
#     world_size=world_size,
# )

# torch.backends.cudnn.benchmark = True

# backbone = get_model(
#     'vit_s', 
#     dropout=0.0, 
#     fp16=True, 
#     num_features=512
# ).cuda()

# backbone = torch.nn.parallel.DistributedDataParallel(
#     module=backbone, 
#     broadcast_buffers=False, 
#     device_ids=[0], 
#     bucket_cap_mb=16,
#     find_unused_parameters=True
# )
# backbone.register_comm_hook(None, fp16_compress_hook)
# backbone._set_static_graph()
# backbone.eval()

# backbone.module.load_state_dict(
#     checkpoint["state_dict_backbone"]
# )

### Test Data Loader

In [76]:
from scripts.utils import get_data_loader, get_embeddings, evaluate

#### agedb 30

In [53]:
agedb30_dataloader = get_data_loader("agedb_30.bin")

In [54]:
embeddings1, embeddings2, issame_list = get_embeddings(agedb30_dataloader, backbone)

Processed batch 1 / 188
Processed batch 2 / 188
Processed batch 3 / 188
Processed batch 4 / 188
Processed batch 5 / 188
Processed batch 6 / 188
Processed batch 7 / 188
Processed batch 8 / 188
Processed batch 9 / 188
Processed batch 10 / 188
Processed batch 11 / 188
Processed batch 12 / 188
Processed batch 13 / 188
Processed batch 14 / 188
Processed batch 15 / 188
Processed batch 16 / 188
Processed batch 17 / 188
Processed batch 18 / 188
Processed batch 19 / 188
Processed batch 20 / 188
Processed batch 21 / 188
Processed batch 22 / 188
Processed batch 23 / 188
Processed batch 24 / 188
Processed batch 25 / 188
Processed batch 26 / 188
Processed batch 27 / 188
Processed batch 28 / 188
Processed batch 29 / 188
Processed batch 30 / 188
Processed batch 31 / 188
Processed batch 32 / 188
Processed batch 33 / 188
Processed batch 34 / 188
Processed batch 35 / 188
Processed batch 36 / 188
Processed batch 37 / 188
Processed batch 38 / 188
Processed batch 39 / 188
Processed batch 40 / 188
Processed

In [55]:
tpr, fpr, accuracy, val, val_std, far = evaluate(
    embeddings1,
    embeddings2,
    issame_list,
    n_folds=10,
    far_target=1e-3,
)

print(f"Accuracy: {accuracy.mean():.4f} ± {accuracy.std():.4f}")
print(f"VAL @ FAR=1e-3: {val:.4f} ± {val_std:.4f}")
print(f"FAR: {far:.6f}")

Accuracy: 0.7577 ± 0.0172
VAL @ FAR=1e-3: 0.0320 ± 0.0150
FAR: 0.001333


### calfw

In [56]:
calfw_dataloader = get_data_loader("calfw.bin")

In [57]:
embeddings1, embeddings2, issame_list = get_embeddings(calfw_dataloader, backbone)

Processed batch 1 / 188
Processed batch 2 / 188
Processed batch 3 / 188
Processed batch 4 / 188
Processed batch 5 / 188
Processed batch 6 / 188
Processed batch 7 / 188
Processed batch 8 / 188
Processed batch 9 / 188
Processed batch 10 / 188
Processed batch 11 / 188
Processed batch 12 / 188
Processed batch 13 / 188
Processed batch 14 / 188
Processed batch 15 / 188
Processed batch 16 / 188
Processed batch 17 / 188
Processed batch 18 / 188
Processed batch 19 / 188
Processed batch 20 / 188
Processed batch 21 / 188
Processed batch 22 / 188
Processed batch 23 / 188
Processed batch 24 / 188
Processed batch 25 / 188
Processed batch 26 / 188
Processed batch 27 / 188
Processed batch 28 / 188
Processed batch 29 / 188
Processed batch 30 / 188
Processed batch 31 / 188
Processed batch 32 / 188
Processed batch 33 / 188
Processed batch 34 / 188
Processed batch 35 / 188
Processed batch 36 / 188
Processed batch 37 / 188
Processed batch 38 / 188
Processed batch 39 / 188
Processed batch 40 / 188
Processed

In [58]:
tpr, fpr, accuracy, val, val_std, far = evaluate(
    embeddings1,
    embeddings2,
    issame_list,
    n_folds=10,
    far_target=1e-3,
)

print(f"Accuracy: {accuracy.mean():.4f} ± {accuracy.std():.4f}")
print(f"VAL @ FAR=1e-3: {val:.4f} ± {val_std:.4f}")
print(f"FAR: {far:.6f}")

Accuracy: 0.8288 ± 0.0105
VAL @ FAR=1e-3: 0.1990 ± 0.0247
FAR: 0.001000


### cfp_ff

In [59]:
cfp_ff_dataloader = get_data_loader("cfp_ff.bin")

In [60]:
embeddings1, embeddings2, issame_list = get_embeddings(cfp_ff_dataloader, backbone)

Processed batch 1 / 219
Processed batch 2 / 219
Processed batch 3 / 219
Processed batch 4 / 219
Processed batch 5 / 219
Processed batch 6 / 219
Processed batch 7 / 219
Processed batch 8 / 219
Processed batch 9 / 219
Processed batch 10 / 219
Processed batch 11 / 219
Processed batch 12 / 219
Processed batch 13 / 219
Processed batch 14 / 219
Processed batch 15 / 219
Processed batch 16 / 219
Processed batch 17 / 219
Processed batch 18 / 219
Processed batch 19 / 219
Processed batch 20 / 219
Processed batch 21 / 219
Processed batch 22 / 219
Processed batch 23 / 219
Processed batch 24 / 219
Processed batch 25 / 219
Processed batch 26 / 219
Processed batch 27 / 219
Processed batch 28 / 219
Processed batch 29 / 219
Processed batch 30 / 219
Processed batch 31 / 219
Processed batch 32 / 219
Processed batch 33 / 219
Processed batch 34 / 219
Processed batch 35 / 219
Processed batch 36 / 219
Processed batch 37 / 219
Processed batch 38 / 219
Processed batch 39 / 219
Processed batch 40 / 219
Processed

In [61]:
tpr, fpr, accuracy, val, val_std, far = evaluate(
    embeddings1,
    embeddings2,
    issame_list,
    n_folds=10,
    far_target=1e-3,
)

print(f"Accuracy: {accuracy.mean():.4f} ± {accuracy.std():.4f}")
print(f"VAL @ FAR=1e-3: {val:.4f} ± {val_std:.4f}")
print(f"FAR: {far:.6f}")

Accuracy: 0.9416 ± 0.0092
VAL @ FAR=1e-3: 0.5749 ± 0.0812
FAR: 0.001143


### cfp_fp

In [62]:
cfp_fp_dataloader = get_data_loader("cfp_fp.bin")

In [63]:
embeddings1, embeddings2, issame_list = get_embeddings(cfp_fp_dataloader, backbone)

Processed batch 1 / 219
Processed batch 2 / 219
Processed batch 3 / 219
Processed batch 4 / 219
Processed batch 5 / 219
Processed batch 6 / 219
Processed batch 7 / 219
Processed batch 8 / 219
Processed batch 9 / 219
Processed batch 10 / 219
Processed batch 11 / 219
Processed batch 12 / 219
Processed batch 13 / 219
Processed batch 14 / 219
Processed batch 15 / 219
Processed batch 16 / 219
Processed batch 17 / 219
Processed batch 18 / 219
Processed batch 19 / 219
Processed batch 20 / 219
Processed batch 21 / 219
Processed batch 22 / 219
Processed batch 23 / 219
Processed batch 24 / 219
Processed batch 25 / 219
Processed batch 26 / 219
Processed batch 27 / 219
Processed batch 28 / 219
Processed batch 29 / 219
Processed batch 30 / 219
Processed batch 31 / 219
Processed batch 32 / 219
Processed batch 33 / 219
Processed batch 34 / 219
Processed batch 35 / 219
Processed batch 36 / 219
Processed batch 37 / 219
Processed batch 38 / 219
Processed batch 39 / 219
Processed batch 40 / 219
Processed

In [64]:
tpr, fpr, accuracy, val, val_std, far = evaluate(
    embeddings1,
    embeddings2,
    issame_list,
    n_folds=10,
    far_target=1e-3,
)

print(f"Accuracy: {accuracy.mean():.4f} ± {accuracy.std():.4f}")
print(f"VAL @ FAR=1e-3: {val:.4f} ± {val_std:.4f}")
print(f"FAR: {far:.6f}")

Accuracy: 0.7356 ± 0.0177
VAL @ FAR=1e-3: 0.0383 ± 0.0130
FAR: 0.001143


### cplfw

In [65]:
cplfw_dataloader = get_data_loader("cplfw.bin")

In [66]:
embeddings1, embeddings2, issame_list = get_embeddings(cplfw_dataloader, backbone)

Processed batch 1 / 188
Processed batch 2 / 188
Processed batch 3 / 188
Processed batch 4 / 188
Processed batch 5 / 188
Processed batch 6 / 188
Processed batch 7 / 188
Processed batch 8 / 188
Processed batch 9 / 188
Processed batch 10 / 188
Processed batch 11 / 188
Processed batch 12 / 188
Processed batch 13 / 188
Processed batch 14 / 188
Processed batch 15 / 188
Processed batch 16 / 188
Processed batch 17 / 188
Processed batch 18 / 188
Processed batch 19 / 188
Processed batch 20 / 188
Processed batch 21 / 188
Processed batch 22 / 188
Processed batch 23 / 188
Processed batch 24 / 188
Processed batch 25 / 188
Processed batch 26 / 188
Processed batch 27 / 188
Processed batch 28 / 188
Processed batch 29 / 188
Processed batch 30 / 188
Processed batch 31 / 188
Processed batch 32 / 188
Processed batch 33 / 188
Processed batch 34 / 188
Processed batch 35 / 188
Processed batch 36 / 188
Processed batch 37 / 188
Processed batch 38 / 188
Processed batch 39 / 188
Processed batch 40 / 188
Processed

In [67]:
tpr, fpr, accuracy, val, val_std, far = evaluate(
    embeddings1,
    embeddings2,
    issame_list,
    n_folds=10,
    far_target=1e-3,
)

print(f"Accuracy: {accuracy.mean():.4f} ± {accuracy.std():.4f}")
print(f"VAL @ FAR=1e-3: {val:.4f} ± {val_std:.4f}")
print(f"FAR: {far:.6f}")

Accuracy: 0.6517 ± 0.0223
VAL @ FAR=1e-3: 0.0047 ± 0.0048
FAR: 0.001333


### lfw

In [77]:
lfw_dataloader = get_data_loader("lfw.bin", images_are_bytes=True)

In [79]:
embeddings1, embeddings2, issame_list = get_embeddings(lfw_dataloader, backbone)

Processed batch 1 / 188
Processed batch 2 / 188
Processed batch 3 / 188
Processed batch 4 / 188
Processed batch 5 / 188
Processed batch 6 / 188
Processed batch 7 / 188
Processed batch 8 / 188
Processed batch 9 / 188
Processed batch 10 / 188
Processed batch 11 / 188
Processed batch 12 / 188
Processed batch 13 / 188
Processed batch 14 / 188
Processed batch 15 / 188
Processed batch 16 / 188
Processed batch 17 / 188
Processed batch 18 / 188
Processed batch 19 / 188
Processed batch 20 / 188
Processed batch 21 / 188
Processed batch 22 / 188
Processed batch 23 / 188
Processed batch 24 / 188
Processed batch 25 / 188
Processed batch 26 / 188
Processed batch 27 / 188
Processed batch 28 / 188
Processed batch 29 / 188
Processed batch 30 / 188
Processed batch 31 / 188
Processed batch 32 / 188
Processed batch 33 / 188
Processed batch 34 / 188
Processed batch 35 / 188
Processed batch 36 / 188
Processed batch 37 / 188
Processed batch 38 / 188
Processed batch 39 / 188
Processed batch 40 / 188
Processed

In [80]:
tpr, fpr, accuracy, val, val_std, far = evaluate(
    embeddings1,
    embeddings2,
    issame_list,
    n_folds=10,
    far_target=1e-3,
)

print(f"Accuracy: {accuracy.mean():.4f} ± {accuracy.std():.4f}")
print(f"VAL @ FAR=1e-3: {val:.4f} ± {val_std:.4f}")
print(f"FAR: {far:.6f}")

Accuracy: 0.9472 ± 0.0129
VAL @ FAR=1e-3: 0.7543 ± 0.0441
FAR: 0.001333
